# Global Ecological Overshoot: A Data Story

**Project question:** Do wealthier and more energy-intensive countries tend to have larger ecological deficits?

This notebook is designed for a sophomore-level data science portfolio project. It combines environmental footprint data with sustainable energy indicators, then uses visualizations and a simple optional model to tell a clear story about global sustainability.

**Primary dataset:** 2016 Global Ecological Footprint  
**Optional supporting dataset:** Global Data on Sustainable Energy 2000-2020

If you only have the ecological footprint dataset, the first half of the notebook still works. If you add the energy dataset too, the notebook builds a deeper merged analysis.


## 1. Setup

Run the install cell only if your notebook environment is missing packages. Kaggle and Google Colab usually already include most of these.


In [ ]:
# If an import fails, uncomment and run this line once:
# %pip install pandas numpy matplotlib seaborn scikit-learn


In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.titleweight"] = "bold"

TARGET_YEAR = 2016


## 2. Data Loading

Put your Kaggle CSV files in one of these places:

- The same folder as this notebook
- A `data/` folder next to this notebook
- A Kaggle input folder such as `/kaggle/input/...`

The loader below searches for CSV files and tries to identify the ecological footprint and sustainable energy datasets automatically.


In [ ]:
def discover_csv_files():
    search_roots = [Path.cwd(), Path.cwd() / "data", Path("/kaggle/input")]
    files = []
    for root in search_roots:
        if root.exists():
            files.extend(root.rglob("*.csv"))
    return sorted(set(files))


def score_file(path, keywords):
    text = str(path).lower()
    return sum(1 for word in keywords if word in text)


def choose_csv(csv_files, keywords):
    if not csv_files:
        return None
    scored = sorted(
        [(score_file(path, keywords), path) for path in csv_files],
        key=lambda item: (item[0], -len(str(item[1]))),
        reverse=True,
    )
    return scored[0][1] if scored[0][0] > 0 else None


csv_files = discover_csv_files()
print(f"Found {len(csv_files)} CSV file(s).")
for path in csv_files[:10]:
    print("-", path)

eco_path = choose_csv(csv_files, ["ecological", "footprint", "biocapacity"])
energy_path = choose_csv(csv_files, ["sustainable", "energy", "renewable"])

print("\nSelected ecological footprint file:", eco_path)
print("Selected sustainable energy file:", energy_path)


In [ ]:
def make_demo_ecological_data():
    return pd.DataFrame({
        "Country": [
            "United States", "China", "India", "Brazil", "Canada", "Germany",
            "Australia", "Japan", "Nigeria", "Norway", "Indonesia", "Mexico"
        ],
        "Region": [
            "North America", "Asia-Pacific", "Asia-Pacific", "Latin America",
            "North America", "Europe", "Asia-Pacific", "Asia-Pacific",
            "Africa", "Europe", "Asia-Pacific", "Latin America"
        ],
        "Total Ecological Footprint": [8.1, 3.6, 1.2, 2.9, 7.8, 5.3, 6.6, 4.7, 1.0, 5.0, 1.7, 2.6],
        "Total Biocapacity": [3.6, 1.0, 0.5, 8.7, 14.9, 1.7, 16.6, 0.6, 1.2, 6.8, 1.4, 1.3],
        "Carbon Footprint": [5.4, 2.6, 0.6, 1.0, 4.7, 3.2, 4.0, 3.2, 0.3, 3.1, 0.8, 1.5],
        "GDP per Capita": [57800, 8123, 1732, 8712, 42158, 42178, 49928, 38900, 2176, 70460, 3562, 8750],
        "HDI": [0.92, 0.75, 0.64, 0.76, 0.93, 0.94, 0.94, 0.91, 0.54, 0.95, 0.69, 0.77],
    })


def make_demo_energy_data():
    return pd.DataFrame({
        "Country": [
            "United States", "China", "India", "Brazil", "Canada", "Germany",
            "Australia", "Japan", "Nigeria", "Norway", "Indonesia", "Mexico"
        ],
        "Year": [2016] * 12,
        "Renewable energy share in the total final energy consumption (%)": [9.2, 12.4, 36.0, 45.2, 22.0, 14.2, 8.4, 6.5, 82.0, 57.8, 36.9, 9.8],
        "Primary energy consumption per capita (kWh/person)": [80000, 26000, 7000, 16000, 103000, 46000, 65000, 42000, 3000, 89000, 8500, 21000],
        "GDP per capita": [57800, 8123, 1732, 8712, 42158, 42178, 49928, 38900, 2176, 70460, 3562, 8750],
        "Value_co2_emissions_kt_by_country": [5000000, 9500000, 2300000, 450000, 550000, 800000, 400000, 1200000, 100000, 45000, 500000, 480000],
        "Energy intensity level of primary energy (MJ/$2017 PPP GDP)": [5.0, 6.1, 4.2, 3.8, 7.4, 3.7, 5.2, 3.6, 4.9, 4.1, 4.6, 4.0],
        "Access to electricity (% of population)": [100, 100, 85, 99, 100, 100, 100, 100, 60, 100, 97, 100],
    })


if eco_path is None:
    print("No ecological footprint CSV found. Using small DEMO data so the notebook can run.")
    raw_eco = make_demo_ecological_data()
    DEMO_MODE = True
else:
    raw_eco = pd.read_csv(eco_path)
    DEMO_MODE = False

if energy_path is None:
    print("No sustainable energy CSV found. Using small DEMO energy data.")
    raw_energy = make_demo_energy_data()
else:
    raw_energy = pd.read_csv(energy_path)

display(raw_eco.head())
display(raw_energy.head())


## 3. Cleaning Helpers

Kaggle column names are not always consistent. These helper functions standardize names and search for likely columns.


In [ ]:
def clean_col_name(name):
    name = str(name).strip().lower()
    name = name.replace("%", " percent ")
    name = re.sub(r"[^a-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name


def normalize_columns(df):
    df = df.copy()
    cleaned = []
    seen = {}
    for col in df.columns:
        base = clean_col_name(col)
        if base in seen:
            seen[base] += 1
            cleaned.append(f"{base}_{seen[base]}")
        else:
            seen[base] = 0
            cleaned.append(base)
    df.columns = cleaned
    return df


def find_col(df, candidates, required_words=None):
    cols = list(df.columns)
    for candidate in candidates:
        cleaned = clean_col_name(candidate)
        if cleaned in cols:
            return cleaned

    if required_words:
        required_words = [clean_col_name(word) for word in required_words]
        for col in cols:
            if all(word in col for word in required_words):
                return col
    return None


def numericify(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip(),
        errors="coerce",
    )


def clean_country_name(value):
    value = str(value).strip()
    value = re.sub(r"\s*\(.*?\)", "", value)
    replacements = {
        "United States of America": "United States",
        "USA": "United States",
        "Russian Federation": "Russia",
        "Viet Nam": "Vietnam",
        "Korea, Rep.": "South Korea",
        "Korea, Dem. People's Rep.": "North Korea",
        "Iran, Islamic Rep.": "Iran",
        "Egypt, Arab Rep.": "Egypt",
    }
    return replacements.get(value, value)


## 4. Clean the Ecological Footprint Data

The key metric is **ecological deficit**:

`ecological deficit = total ecological footprint - total biocapacity`

Positive values mean the country uses more ecological resources than its ecosystems can regenerate. Negative values mean an ecological reserve.


In [ ]:
eco = normalize_columns(raw_eco)
print("Ecological columns:")
print(list(eco.columns))

country_col = find_col(eco, ["country", "country_name", "nation"], ["country"])
region_col = find_col(eco, ["region", "continent"], ["region"])
footprint_col = find_col(
    eco,
    ["total_ecological_footprint", "ecological_footprint", "total_footprint"],
    ["ecological", "footprint"],
)
biocapacity_col = find_col(
    eco,
    ["total_biocapacity", "biocapacity"],
    ["biocapacity"],
)
carbon_col = find_col(eco, ["carbon_footprint", "carbon"], ["carbon", "footprint"])
gdp_eco_col = find_col(eco, ["gdp_per_capita", "gdp"], ["gdp"])
hdi_col = find_col(eco, ["hdi", "human_development_index"], ["hdi"])

needed = {
    "country": country_col,
    "total_ecological_footprint": footprint_col,
    "total_biocapacity": biocapacity_col,
}
missing = [name for name, col in needed.items() if col is None]
if missing:
    raise ValueError(f"Missing needed ecological columns: {missing}. Check column names above.")

eco_clean = pd.DataFrame({
    "country": eco[country_col].apply(clean_country_name),
    "total_ecological_footprint": numericify(eco[footprint_col]),
    "total_biocapacity": numericify(eco[biocapacity_col]),
})

if region_col:
    eco_clean["region"] = eco[region_col].astype(str)
else:
    eco_clean["region"] = "Unknown"

if carbon_col:
    eco_clean["carbon_footprint"] = numericify(eco[carbon_col])
if gdp_eco_col:
    eco_clean["gdp_per_capita_eco"] = numericify(eco[gdp_eco_col])
if hdi_col:
    eco_clean["hdi"] = numericify(eco[hdi_col])

eco_clean = eco_clean.dropna(subset=["country", "total_ecological_footprint", "total_biocapacity"])
eco_clean["ecological_deficit"] = eco_clean["total_ecological_footprint"] - eco_clean["total_biocapacity"]
eco_clean["ecological_reserve"] = -eco_clean["ecological_deficit"]
eco_clean["status"] = np.where(eco_clean["ecological_deficit"] > 0, "Deficit", "Reserve")

if "carbon_footprint" in eco_clean:
    eco_clean["carbon_share_of_footprint"] = eco_clean["carbon_footprint"] / eco_clean["total_ecological_footprint"]

eco_clean = eco_clean.sort_values("ecological_deficit", ascending=False)
display(eco_clean.head(10))


## 5. First Findings: Who Is in Deficit?

These charts establish the main environmental story before adding energy or economic variables.


In [ ]:
summary = eco_clean["status"].value_counts()
deficit_share = (eco_clean["status"].eq("Deficit").mean()) * 100

print(summary)
print(f"\n{deficit_share:.1f}% of countries in this dataset are in ecological deficit.")

plt.figure(figsize=(7, 5))
sns.countplot(data=eco_clean, x="status", order=["Deficit", "Reserve"], palette=["#c7524a", "#2f8f83"])
plt.title("Countries by Ecological Status")
plt.xlabel("")
plt.ylabel("Number of countries")
plt.show()


In [ ]:
top_deficit = eco_clean.nlargest(15, "ecological_deficit")

plt.figure(figsize=(11, 7))
sns.barplot(
    data=top_deficit,
    y="country",
    x="ecological_deficit",
    hue="region",
    dodge=False,
    palette="tab10",
)
plt.title("Top 15 Countries With the Largest Ecological Deficits")
plt.xlabel("Ecological deficit per person")
plt.ylabel("")
plt.legend(title="Region", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
top_reserve = eco_clean.nsmallest(15, "ecological_deficit").copy()
top_reserve["reserve_amount"] = -top_reserve["ecological_deficit"]

plt.figure(figsize=(11, 7))
sns.barplot(
    data=top_reserve.sort_values("reserve_amount", ascending=False),
    y="country",
    x="reserve_amount",
    hue="region",
    dodge=False,
    palette="tab10",
)
plt.title("Top 15 Countries With the Largest Ecological Reserves")
plt.xlabel("Ecological reserve per person")
plt.ylabel("")
plt.legend(title="Region", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9, 7))
sns.scatterplot(
    data=eco_clean,
    x="total_biocapacity",
    y="total_ecological_footprint",
    hue="status",
    style="status",
    palette={"Deficit": "#c7524a", "Reserve": "#2f8f83"},
    s=90,
)

limit = max(eco_clean["total_biocapacity"].max(), eco_clean["total_ecological_footprint"].max())
plt.plot([0, limit], [0, limit], color="black", linestyle="--", linewidth=1)
plt.title("Ecological Footprint vs. Biocapacity")
plt.xlabel("Biocapacity per person")
plt.ylabel("Ecological footprint per person")
plt.text(limit * 0.55, limit * 0.92, "Above line = deficit", fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
if "carbon_share_of_footprint" in eco_clean:
    carbon_view = (
        eco_clean
        .dropna(subset=["carbon_share_of_footprint"])
        .query("total_ecological_footprint > 0")
        .nlargest(15, "carbon_share_of_footprint")
        .copy()
    )
    carbon_view["carbon_share_percent"] = carbon_view["carbon_share_of_footprint"] * 100

    plt.figure(figsize=(11, 7))
    sns.barplot(data=carbon_view, y="country", x="carbon_share_percent", color="#5b7c99")
    plt.title("Where Carbon Makes Up the Largest Share of Ecological Footprint")
    plt.xlabel("Carbon share of ecological footprint (%)")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()
else:
    print("No carbon footprint column found, so this chart is skipped.")


## 6. Clean and Merge the Energy Dataset

This section adds energy and economic context. The project becomes stronger if you include the sustainable energy dataset, but the notebook can still work without it.


In [ ]:
energy = normalize_columns(raw_energy)
print("Energy columns:")
print(list(energy.columns))

country_energy_col = find_col(energy, ["country", "country_name", "entity"], ["country"])
year_col = find_col(energy, ["year"], ["year"])
renewable_col = find_col(
    energy,
    [
        "renewable_energy_share_in_the_total_final_energy_consumption_percent",
        "renewable_energy_share",
        "renewables_percent_equivalent_primary_energy",
    ],
    ["renewable"],
)
primary_energy_col = find_col(
    energy,
    ["primary_energy_consumption_per_capita_kwh_person", "primary_energy_consumption_per_capita"],
    ["energy", "per", "capita"],
)
co2_col = find_col(
    energy,
    ["value_co2_emissions_kt_by_country", "co2_emissions_kt", "co2_emissions"],
    ["co2"],
)
gdp_energy_col = find_col(energy, ["gdp_per_capita"], ["gdp", "per", "capita"])
energy_intensity_col = find_col(
    energy,
    ["energy_intensity_level_of_primary_energy_mj_2017_ppp_gdp", "energy_intensity"],
    ["energy", "intensity"],
)
electricity_access_col = find_col(
    energy,
    ["access_to_electricity_percent_of_population", "access_to_electricity"],
    ["electricity"],
)

if country_energy_col is None:
    print("No country column found in the energy data, so the merge will be skipped.")
    energy_clean = pd.DataFrame(columns=["country"])
else:
    energy_clean = energy.copy()
    energy_clean["country"] = energy_clean[country_energy_col].apply(clean_country_name)

    if year_col:
        energy_clean[year_col] = numericify(energy_clean[year_col])
        available_years = sorted(energy_clean[year_col].dropna().unique())
        if TARGET_YEAR in available_years:
            selected_year = TARGET_YEAR
        else:
            selected_year = min(available_years, key=lambda year: abs(year - TARGET_YEAR))
        print(f"Using energy data from {int(selected_year)}.")
        energy_clean = energy_clean.loc[energy_clean[year_col].eq(selected_year)].copy()
    else:
        print("No year column found in energy data.")

    keep = ["country"]
    rename_map = {}

    optional_cols = {
        "renewable_energy_share": renewable_col,
        "primary_energy_per_capita": primary_energy_col,
        "co2_emissions_kt": co2_col,
        "gdp_per_capita_energy": gdp_energy_col,
        "energy_intensity": energy_intensity_col,
        "electricity_access": electricity_access_col,
    }

    for clean_name, col in optional_cols.items():
        if col:
            keep.append(col)
            rename_map[col] = clean_name

    energy_clean = energy_clean[keep].rename(columns=rename_map)
    for col in energy_clean.columns:
        if col != "country":
            energy_clean[col] = numericify(energy_clean[col])

display(energy_clean.head())


In [ ]:
merged = eco_clean.merge(energy_clean, on="country", how="inner")

print(f"Ecological rows: {len(eco_clean)}")
print(f"Merged rows: {len(merged)}")

if len(merged) == 0:
    print("No matched countries. Check country names or use only the ecological footprint analysis.")
else:
    display(merged.head())


## 7. Energy, Wealth, and Environmental Pressure

These charts answer the deeper project question: whether economic and energy variables appear connected to ecological overshoot.


In [ ]:
if len(merged) > 0 and "gdp_per_capita_energy" in merged:
    plt.figure(figsize=(10, 6))
    sns.scatterplot(
        data=merged,
        x="gdp_per_capita_energy",
        y="ecological_deficit",
        hue="status",
        palette={"Deficit": "#c7524a", "Reserve": "#2f8f83"},
        s=90,
    )
    plt.axhline(0, color="black", linestyle="--", linewidth=1)
    plt.xscale("log")
    plt.title("GDP per Capita vs. Ecological Deficit")
    plt.xlabel("GDP per capita, log scale")
    plt.ylabel("Ecological deficit per person")
    plt.tight_layout()
    plt.show()
else:
    print("GDP per capita was not available in the merged data.")


In [ ]:
if len(merged) > 0 and "primary_energy_per_capita" in merged:
    plt.figure(figsize=(10, 6))
    sns.scatterplot(
        data=merged,
        x="primary_energy_per_capita",
        y="ecological_deficit",
        hue="status",
        palette={"Deficit": "#c7524a", "Reserve": "#2f8f83"},
        s=90,
    )
    plt.axhline(0, color="black", linestyle="--", linewidth=1)
    plt.xscale("log")
    plt.title("Energy Consumption per Person vs. Ecological Deficit")
    plt.xlabel("Primary energy consumption per capita, log scale")
    plt.ylabel("Ecological deficit per person")
    plt.tight_layout()
    plt.show()
else:
    print("Primary energy consumption per capita was not available in the merged data.")


In [ ]:
if len(merged) > 0 and {"renewable_energy_share", "co2_emissions_kt"}.issubset(merged.columns):
    plt.figure(figsize=(10, 6))
    sns.scatterplot(
        data=merged,
        x="renewable_energy_share",
        y="co2_emissions_kt",
        hue="status",
        palette={"Deficit": "#c7524a", "Reserve": "#2f8f83"},
        s=90,
    )
    plt.yscale("log")
    plt.title("Renewable Energy Share vs. Total CO2 Emissions")
    plt.xlabel("Renewable energy share (%)")
    plt.ylabel("CO2 emissions, kt, log scale")
    plt.tight_layout()
    plt.show()
else:
    print("Renewable energy share and CO2 emissions were not both available.")


In [ ]:
if len(merged) > 0:
    numeric_cols = [
        col for col in [
            "ecological_deficit",
            "total_ecological_footprint",
            "total_biocapacity",
            "carbon_footprint",
            "carbon_share_of_footprint",
            "renewable_energy_share",
            "primary_energy_per_capita",
            "co2_emissions_kt",
            "gdp_per_capita_energy",
            "energy_intensity",
            "electricity_access",
            "hdi",
        ]
        if col in merged.columns
    ]

    corr = merged[numeric_cols].corr(numeric_only=True)
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, linewidths=0.5)
    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.show()
else:
    print("No merged data available for correlation analysis.")


## 8. Optional Simple Model

This is not the main point of the project, but a small model can help you discuss which variables are most useful for predicting ecological deficit.


In [ ]:
try:
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import mean_absolute_error, r2_score

    model_features = [
        col for col in [
            "renewable_energy_share",
            "primary_energy_per_capita",
            "co2_emissions_kt",
            "gdp_per_capita_energy",
            "energy_intensity",
            "electricity_access",
            "hdi",
        ]
        if col in merged.columns
    ]

    model_data = merged[["ecological_deficit"] + model_features].dropna()

    if len(model_data) >= 25 and len(model_features) >= 2:
        X = model_data[model_features]
        y = model_data["ecological_deficit"]

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=42
        )

        model = RandomForestRegressor(n_estimators=300, random_state=42)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        print(f"Mean absolute error: {mean_absolute_error(y_test, preds):.2f}")
        print(f"R-squared: {r2_score(y_test, preds):.2f}")

        importance = (
            pd.DataFrame({"feature": model_features, "importance": model.feature_importances_})
            .sort_values("importance", ascending=False)
        )

        plt.figure(figsize=(9, 5))
        sns.barplot(data=importance, y="feature", x="importance", color="#587c70")
        plt.title("Feature Importance for Predicting Ecological Deficit")
        plt.xlabel("Importance")
        plt.ylabel("")
        plt.tight_layout()
        plt.show()
    else:
        print("Not enough complete merged rows/features for a useful model.")
        print(f"Rows available: {len(model_data)}")
        print(f"Features available: {model_features}")

except ImportError:
    print("scikit-learn is not installed. Run the install cell if you want the optional model.")


## 9. Auto-Generated Findings

Use these outputs as a starting point for your written conclusion. Rewrite them in your own voice before submitting.


In [ ]:
findings = []

if len(eco_clean) > 0:
    biggest_deficit = eco_clean.iloc[0]
    biggest_reserve = eco_clean.iloc[-1]
    findings.append(
        f"{biggest_deficit['country']} has the largest ecological deficit in this dataset."
    )
    findings.append(
        f"{biggest_reserve['country']} has the largest ecological reserve in this dataset."
    )
    findings.append(
        f"{deficit_share:.1f}% of countries are classified as ecological deficit countries."
    )

if len(merged) > 0 and "gdp_per_capita_energy" in merged:
    corr_gdp = merged[["gdp_per_capita_energy", "ecological_deficit"]].corr().iloc[0, 1]
    findings.append(
        f"The correlation between GDP per capita and ecological deficit is {corr_gdp:.2f}."
    )

if len(merged) > 0 and "primary_energy_per_capita" in merged:
    corr_energy = merged[["primary_energy_per_capita", "ecological_deficit"]].corr().iloc[0, 1]
    findings.append(
        f"The correlation between energy use per person and ecological deficit is {corr_energy:.2f}."
    )

if "carbon_share_of_footprint" in eco_clean:
    avg_carbon_share = eco_clean["carbon_share_of_footprint"].mean() * 100
    findings.append(
        f"On average, carbon accounts for {avg_carbon_share:.1f}% of ecological footprint in this dataset."
    )

print("Draft findings:")
for idx, finding in enumerate(findings, start=1):
    print(f"{idx}. {finding}")


## 10. Final Write-Up Template

Use this structure for the final project explanation.

### Introduction
This project studies whether countries with higher economic development and energy use are more likely to run ecological deficits. An ecological deficit occurs when a country's ecological footprint is larger than its biocapacity.

### Methods
I cleaned country-level environmental data, calculated ecological deficit, visualized which countries had the largest deficits and reserves, then merged the footprint data with sustainable energy indicators for 2016. I used bar charts, scatter plots, and a correlation heatmap to compare environmental pressure with energy and economic variables.

### Main Findings
1. Replace this with your first chart-based finding.
2. Replace this with your second chart-based finding.
3. Replace this with your third chart-based finding.

### Conclusion
The results suggest that ecological overshoot is connected not only to population, but also to consumption patterns, energy systems, and economic development. Countries with high energy use often place more pressure on ecological resources, while renewable energy and high biocapacity can change the sustainability picture.

### Limitations
This project uses country-level data, so it cannot explain differences within countries. Some datasets may use estimates, and country-name matching can remove countries from the merged analysis. The project should be interpreted as exploratory rather than causal.
